# Learning Rate Scheduling

## Objectives
- Understand StepLR, ExponentialLR, CosineAnnealingLR, ReduceLROnPlateau
- Learn about learning rate warmup
- Visualize different schedules
- Compare their effects on convergence
- Choose appropriate schedules for different tasks

## Introduction
Learning rate scheduling adaptively adjusts the learning rate during training. This can improve convergence speed and final performance. This notebook covers the most important schedules.

## What We're About to Do

The code below imports essential libraries. These libraries provide the foundational tools for tensor operations and neural network construction. Pay attention to what each import provides – understanding dependencies helps you know what's available for solving problems.


In [ ]:
# Import necessary libraries for tensor operations and deep learning
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import (
    StepLR, ExponentialLR, CosineAnnealingLR, ReduceLROnPlateau, 
    LinearLR, PolynomialLR
)
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")


In [ ]:
# Visualize the results
plt.figure(figsize=(12, 4))
plt.plot(range(len(result)), result, marker='o', linestyle='-', linewidth=2)
plt.title('Results Visualization', fontsize=14, fontweight='bold')
plt.xlabel('Index')
plt.ylabel('Value')
plt.grid(True, alpha=0.3)
plt.show()

print('Visualization shows the pattern and distribution of results.')


## The Training Process

This is the core learning loop. We'll forward-pass data through the model, compute loss, backpropagate gradients, and update parameters. This iterative process gradually improves the model.


In [ ]:
# Execute the training loop with proper tracking
## 1. Create Dataset

X, y = make_classification(n_samples=500, n_features=20, n_classes=2,
                            n_informative=15, n_redundant=5, random_state=42)

X_train, X_test = torch.FloatTensor(X[:400]), torch.FloatTensor(X[400:])
y_train, y_test = torch.LongTensor(y[:400]), torch.LongTensor(y[400:])

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=32, shuffle=False)

print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")


## Building the Model

Now we'll define our neural network architecture. Each layer transforms the input in a specific way, building up complexity. The order and configuration of layers directly determines what patterns the model can learn.


In [ ]:
# Define a custom function with detailed implementation
## 2. Define Simple Model

class SimpleClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(20, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 2)
        )
    
    def forward(self, x):
        return self.network(x)

print("Model defined.")


## Building the Model

Now we'll define our neural network architecture. Each layer transforms the input in a specific way, building up complexity. The order and configuration of layers directly determines what patterns the model can learn.


In [ ]:
# Define a custom function with detailed implementation
## 3. Training Function with Scheduler

def train_with_scheduler(scheduler_class, scheduler_kwargs, epochs=100, init_lr=0.01):
    """Train model with learning rate scheduler"""
    model = SimpleClassifier().to(device)
# Update model parameters based on computed gradients
    optimizer = optim.Adam(model.parameters(), lr=init_lr)
# Compute the loss (error) between predictions and actual values
    criterion = nn.CrossEntropyLoss()
    
    # Create scheduler
    if scheduler_class == ReduceLROnPlateau:
        scheduler = scheduler_class(optimizer, **scheduler_kwargs)
    else:
        scheduler = scheduler_class(optimizer, **scheduler_kwargs)
    
# Compute the loss (error) between predictions and actual values
    train_losses = []
    test_accs = []
    lrs = []
    
    X_test_device = X_test.to(device)
    y_test_device = y_test.to(device)
    
# Iterate through batches of data
    for epoch in range(epochs):
        model.train()
# Compute the loss (error) between predictions and actual values
        epoch_loss = 0
        
# Iterate through batches of data
        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            
            logits = model(X_batch)
# Compute the loss (error) between predictions and actual values
            loss = criterion(logits, y_batch)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
# Compute the loss (error) between predictions and actual values
            epoch_loss += loss.item()
        
        train_losses.append(epoch_loss / len(train_loader))
        
        # Evaluate
        model.eval()
        with torch.no_grad():
            logits = model(X_test_device)
            preds = logits.argmax(dim=1)
            acc = (preds == y_test_device).float().mean().item()
            test_accs.append(acc)
        
        # Record LR
        lrs.append(optimizer.param_groups[0]['lr'])
        
        # Step scheduler
        if scheduler_class == ReduceLROnPlateau:
            scheduler.step(epoch_loss / len(train_loader))
        else:
            scheduler.step()
    
    return train_losses, test_accs, lrs

print("Training function defined.")


## The Training Process

This is the core learning loop. We'll forward-pass data through the model, compute loss, backpropagate gradients, and update parameters. This iterative process gradually improves the model.


In [ ]:
# Execute the training loop with proper tracking
## 4. StepLR Schedule

print("Training with StepLR...")
results_steplr = train_with_scheduler(
    StepLR,
    {'step_size': 20, 'gamma': 0.5},
    epochs=100,
    init_lr=0.01
)
print("StepLR completed.")


## The Training Process

This is the core learning loop. We'll forward-pass data through the model, compute loss, backpropagate gradients, and update parameters. This iterative process gradually improves the model.


In [ ]:
# Execute the training loop with proper tracking
## 5. ExponentialLR Schedule

print("Training with ExponentialLR...")
results_exp = train_with_scheduler(
    ExponentialLR,
    {'gamma': 0.95},
    epochs=100,
    init_lr=0.01
)
print("ExponentialLR completed.")


## The Training Process

This is the core learning loop. We'll forward-pass data through the model, compute loss, backpropagate gradients, and update parameters. This iterative process gradually improves the model.


In [ ]:
# Execute the training loop with proper tracking
## 6. CosineAnnealingLR Schedule

print("Training with CosineAnnealingLR...")
results_cosine = train_with_scheduler(
    CosineAnnealingLR,
    {'T_max': 50},
    epochs=100,
    init_lr=0.01
)
print("CosineAnnealingLR completed.")


## Building the Model

Now we'll define our neural network architecture. Each layer transforms the input in a specific way, building up complexity. The order and configuration of layers directly determines what patterns the model can learn.


In [ ]:
# Set up the neural network model architecture
## 7. No Scheduler (Constant LR)

print("Training with constant learning rate...")

model = SimpleClassifier().to(device)
# Update model parameters based on computed gradients
optimizer = optim.Adam(model.parameters(), lr=0.01)
# Compute the loss (error) between predictions and actual values
criterion = nn.CrossEntropyLoss()

# Compute the loss (error) between predictions and actual values
train_losses_const = []
test_accs_const = []
lrs_const = []

X_test_device = X_test.to(device)
y_test_device = y_test.to(device)

# Iterate through batches of data
for epoch in range(100):
    model.train()
# Compute the loss (error) between predictions and actual values
    epoch_loss = 0
    
# Iterate through batches of data
    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        
        logits = model(X_batch)
# Compute the loss (error) between predictions and actual values
        loss = criterion(logits, y_batch)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
# Compute the loss (error) between predictions and actual values
        epoch_loss += loss.item()
    
    train_losses_const.append(epoch_loss / len(train_loader))
    
    model.eval()
    with torch.no_grad():
        logits = model(X_test_device)
        preds = logits.argmax(dim=1)
        acc = (preds == y_test_device).float().mean().item()
        test_accs_const.append(acc)
    
    lrs_const.append(optimizer.param_groups[0]['lr'])

# Compute the loss (error) between predictions and actual values
results_const = (train_losses_const, test_accs_const, lrs_const)
print("Constant LR completed.")


## Defining the Loss Function

The loss function measures how wrong our predictions are. During training, we'll minimize this value. Different tasks need different loss functions – the one we choose defines what 'good performance' means for our model.


In [ ]:
# Configure loss function and optimization algorithm
## 8. Visualize LR Schedules

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Learning rate schedules
ax = axes[0, 0]
ax.plot(results_const[2], label='Constant', linewidth=2, color='gray')
ax.plot(results_steplr[2], label='StepLR', linewidth=2)
ax.plot(results_exp[2], label='ExponentialLR', linewidth=2)
ax.plot(results_cosine[2], label='CosineAnnealingLR', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Learning Rate')
ax.set_title('Learning Rate Schedules')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_yscale('log')

# Training loss
ax = axes[0, 1]
ax.plot(results_const[0], label='Constant', linewidth=2, color='gray')
ax.plot(results_steplr[0], label='StepLR', linewidth=2)
ax.plot(results_exp[0], label='ExponentialLR', linewidth=2)
ax.plot(results_cosine[0], label='CosineAnnealingLR', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Training Loss')
ax.set_title('Training Loss Comparison')
ax.legend()
ax.grid(True, alpha=0.3)

# Test accuracy
ax = axes[1, 0]
ax.plot(results_const[1], label='Constant', linewidth=2, color='gray')
ax.plot(results_steplr[1], label='StepLR', linewidth=2)
ax.plot(results_exp[1], label='ExponentialLR', linewidth=2)
ax.plot(results_cosine[1], label='CosineAnnealingLR', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Test Accuracy')
ax.set_title('Test Accuracy Comparison')
ax.legend()
ax.grid(True, alpha=0.3)

# Final performance comparison
ax = axes[1, 1]
schedules = ['Constant', 'StepLR', 'ExponentialLR', 'CosineAnnealing']
final_accs = [
    results_const[1][-1],
    results_steplr[1][-1],
    results_exp[1][-1],
    results_cosine[1][-1]
]
colors = ['gray', 'C0', 'C1', 'C2']
bars = ax.bar(schedules, final_accs, color=colors)
ax.set_ylabel('Final Test Accuracy')
ax.set_title('Final Performance by Schedule')
ax.set_ylim([0.5, 1.0])
ax.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
# Iterate through batches of data
for bar, acc in zip(bars, final_accs):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{acc:.4f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()


In [ ]:
# Define a custom function with detailed implementation
## 9. Warmup Schedule

# Demonstrate linear warmup + cosine annealing
def visualize_warmup_schedule():
    epochs = 100
    warmup_epochs = 10
    
    # Linear warmup
    warmup_lrs = np.linspace(0, 0.01, warmup_epochs)
    
    # Cosine annealing after warmup
    remaining_epochs = epochs - warmup_epochs
    cosine_lrs = 0.01 * 0.5 * (1 + np.cos(np.pi * np.arange(remaining_epochs) / remaining_epochs))
    
    all_lrs = np.concatenate([warmup_lrs, cosine_lrs])
    
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(all_lrs, linewidth=2.5)
    ax.axvline(x=warmup_epochs, color='r', linestyle='--', linewidth=2, label='Warmup end')
    ax.fill_between(range(warmup_epochs), all_lrs[:warmup_epochs], alpha=0.3, label='Warmup phase')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Learning Rate')
    ax.set_title('Warmup + Cosine Annealing Schedule')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print(f"Warmup phase (0-{warmup_epochs}): LR linearly increases from 0 to 0.01")
    print(f"Annealing phase ({warmup_epochs}-{epochs}): LR cosine anneals from 0.01 to ~0")

visualize_warmup_schedule()


## What We're About to Do

The code below imports essential libraries. These libraries provide the foundational tools for tensor operations and neural network construction. Pay attention to what each import provides – understanding dependencies helps you know what's available for solving problems.


In [ ]:
# Import necessary libraries for tensor operations and deep learning
## 10. Scheduler Selection Guide

import pandas as pd

scheduler_guide = {
    'Scheduler': ['Constant', 'StepLR', 'ExponentialLR', 'CosineAnnealingLR', 'ReduceLROnPlateau', 'Linear Warmup'],
    'Formula/Behavior': [
        'No change',
        'lr = lr * gamma every step_size epochs',
        'lr = lr * gamma every epoch',
        'Cosine annealing over T_max epochs',
        'Reduce on plateau in validation metric',
        'Linear increase from 0 to target'
    ],
    'Best For': [
        'Simple models, fine-tuning',
        'Standard training, periodic decay',
        'Smooth gradual decay',
        'Modern architectures, transformers',
        'Adaptive reduction, any model',
        'Large batch training, transformers'
    ],
    'Key Parameters': [
        'None',
        'step_size, gamma',
        'gamma',
        'T_max, eta_min',
        'factor, patience',
        'total_iters'
    ]
}

df = pd.DataFrame(scheduler_guide)
print(df.to_string(index=False))


## What We're About to Do

The code below imports essential libraries. These libraries provide the foundational tools for tensor operations and neural network construction. Pay attention to what each import provides – understanding dependencies helps you know what's available for solving problems.


In [ ]:
# Import necessary libraries for tensor operations and deep learning
## 11. Best Practices Summary

print("\n" + "="*70)
print("Learning Rate Scheduling Best Practices")
print("="*70)

best_practices = """
1. START WITH CONSTANT LR:
   - Establish baseline performance
   - Tune initial learning rate value

2. CHOOSE SCHEDULE TYPE:
   - Transformers: Linear warmup + cosine annealing
   - CNNs: StepLR or ExponentialLR
   - Adaptive: ReduceLROnPlateau

3. COMMON SCHEDULES:
   - Step: LR *= 0.1 every 30 epochs
   - Exponential: LR *= 0.95 every epoch
   - Cosine: Smooth decay from max to min
   - Warmup: Linear from 0 to target over 5-10% of training

4. HYPERPARAMETERS:
   - Initial LR: Most important, tune first
   - Gamma (decay): 0.1 (step decay) or 0.95 (exponential)
   - Step size: 1/3 to 1/2 of total epochs
   - Warmup: 5-10% of total epochs

5. MONITORING:
   - Track actual LR changes during training
# Iterate through batches of data
   - Watch for LR dropping too quickly
   - Ensure LR reaches small values by end of training

6. COMBINATIONS:
# Iterate through batches of data
   - Warmup + StepLR: Good for large batch training
   - Warmup + Cosine: Excellent for transformers
   - ReduceLROnPlateau: Good safety net
"""

print(best_practices)
print("="*70)


## What We're About to Do

The code below imports essential libraries. These libraries provide the foundational tools for tensor operations and neural network construction. Pay attention to what each import provides – understanding dependencies helps you know what's available for solving problems.


## 🎯 Key Takeaways

✅ **Understanding fundamentals is crucial** – The concepts covered here form the foundation for all advanced deep learning techniques.

✅ **Each component has a specific purpose** – Whether it's data loading, model architecture, or optimization, every piece serves a distinct function in the pipeline.

✅ **Experimentation drives learning** – Don't just read the code; modify it, break it, and see what happens. That's how intuition develops.

✅ **Deep learning is iterative** – Success comes from systematically trying approaches, measuring results, and refining based on evidence.

✅ **Connect concepts, don't memorize** – Understanding how PyTorch tensors relate to autograd, which relates to neural networks, which connects to training loops, is far more valuable than memorizing individual APIs.

✅ **Performance matters in practice** – Once you understand the theory, optimizing for speed, memory, and scalability becomes crucial for real-world applications.


In [ ]:
# Import necessary libraries for tensor operations and deep learning
## Key Takeaways
- Learning rate scheduling can significantly improve training
# Iterate through batches of data
- CosineAnnealingLR is effective for modern architectures
- Warmup is important for transformer training
- StepLR provides simple, effective learning rate decay
- Different schedules suit different architectures

## Interview Q&A

**Q1: Why is warmup important?**
Large learning rates early in training can cause instability and divergence. Warmup gradually increases the learning rate, allowing the model to find a stable region before accelerating training.

**Q2: How does CosineAnnealingLR compare to StepLR?**
# Iterate through batches of data
StepLR has discrete drops at specific epochs. CosineAnnealingLR provides smooth, continuous decay. Cosine is generally better for modern architectures and requires less tuning.

**Q3: When should you use ReduceLROnPlateau?**
Use ReduceLROnPlateau when you don't know the training duration in advance or when validation performance plateaus. It's adaptive but requires more tuning than fixed schedules.

## References
- [PyTorch Learning Rate Schedulers](https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate)
- [Cosine Annealing Paper](https://arxiv.org/abs/1608.03983)
- [Transformer Training Best Practices](https://arxiv.org/abs/2004.08249)
